In [16]:
import json
import random
import re
import time
from datetime import date, datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from renewables_permitting.utils import (
    normalize_text,
    save_parquet,
    validate_required_columns,
)

import hashlib

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# BRONZE
BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"


# SILVER
BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"

BOE_CANDIDATES_DOCS_TEXT_PATH = SILVER_DIR / "boe_candidates_docs_text" / "boe_candidates_docs_text.parquet"

DIM_MUNICIPALITIES_PATH = SILVER_DIR / "dimensions" / "dim_municipalities.parquet"

SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"
LIFECYCLE_EVENTS_PATH = SILVER_BOE_AI_DIR / "lifecycle_events.parquet"
ADMINISTRATIVE_ACTIONS_PATH = SILVER_BOE_AI_DIR / "administrative_actions.parquet"
ASSET_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_mentions.parquet"
ASSET_TECHNOLOGIES_PATH = SILVER_BOE_AI_DIR / "asset_technologies.parquet"
ASSET_PARTICIPANTS_PATH = SILVER_BOE_AI_DIR / "asset_participants.parquet"
ASSET_LOCATIONS_PATH = SILVER_BOE_AI_DIR / "asset_locations.parquet"
ASSET_ALIASES_PATH = SILVER_BOE_AI_DIR / "asset_aliases.parquet"
ASSET_RELATION_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_relation_mentions.parquet"


# GOLD
PROJECT_ASSET_MENTIONS_PATH = GOLD_DIR / "project_asset_mentions.parquet"
PROJECT_GROUPS_PATH = GOLD_DIR / "project_groups.parquet"

PROJECT_ASSETS_PATH = GOLD_DIR / "project_assets.parquet"
PROJECT_TIMELINE_PATH = GOLD_DIR / "project_timeline.parquet"
PROJECT_STATUS_PATH = GOLD_DIR / "project_status.parquet"

## Crear una clave geográfica por activo

In [ ]:
asset_mentions = pd.read_parquet(ASSET_MENTIONS_PATH)
asset_locations = pd.read_parquet(ASSET_LOCATIONS_PATH)

In [ ]:
def normalize_asset_name_for_grouping(
    text: str | None,
) -> str:
    """
    Normaliza nombres de activos energéticos para agrupación.

    Elimina términos genéricos frecuentes que pueden variar entre BOE:
    PE, parque, parque eólico, planta, instalación, etc.
    """
    text = normalize_text(text)

    generic_tokens = {
        "pe",
        "pfv",
        "fv",
        "parque",
        "eolico",
        "planta",
        "fotovoltaica",
        "solar",
        "instalacion",
        "instalaciones",
    }

    tokens = [
        token
        for token in text.split()
        if token not in generic_tokens
    ]

    return " ".join(tokens)

In [19]:
def build_asset_location_keys(
    asset_locations: pd.DataFrame,
) -> pd.DataFrame:
    """
    Construye una clave territorial estable por mención de activo.

    La clave se basa en el conjunto ordenado de códigos INE de municipio
    asociados a cada asset_mention_id.
    """
    required_cols = {
        "asset_mention_id",
        "ine_municipality_code",
    }
    validate_required_columns(asset_locations, required_cols)

    location_keys = (
        asset_locations
        .dropna(subset=["asset_mention_id", "ine_municipality_code"])
        .assign(
            ine_municipality_code=lambda df: (
                df["ine_municipality_code"].astype(str)
            )
        )
        .groupby("asset_mention_id", as_index=False)
        .agg(
            municipality_codes_key=(
                "ine_municipality_code",
                lambda values: "|".join(sorted(set(values))),
            ),
            n_municipalities=(
                "ine_municipality_code",
                lambda values: len(set(values)),
            ),
        )
    )

    return location_keys

In [20]:
def build_project_candidates(
    asset_mentions: pd.DataFrame,
    asset_location_keys: pd.DataFrame,
) -> pd.DataFrame:
    """
    Construye candidatos de agrupación proyecto-activo.

    La clave inicial de agrupación es:
    asset_name_norm + conjunto de municipios INE.
    """
    required_asset_cols = {
        "asset_mention_id",
        "event_id",
        "identificador_boe",
        "fecha_publicacion",
        "asset_name",
        "asset_name_norm",
    }
    required_location_cols = {
        "asset_mention_id",
        "municipality_codes_key",
    }

    validate_required_columns(asset_mentions, required_asset_cols)
    validate_required_columns(asset_location_keys, required_location_cols)

    candidates = asset_mentions.merge(
        asset_location_keys,
        on="asset_mention_id",
        how="left",
    )

    candidates["asset_name_norm"] = (
        candidates["asset_name_norm"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    candidates["municipality_codes_key"] = (
        candidates["municipality_codes_key"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    candidates["project_match_key"] = (
        candidates["asset_name_norm"]
        + "__"
        + candidates["municipality_codes_key"]
    )

    candidates = candidates.loc[
        (candidates["asset_name_norm"] != "")
        & (candidates["municipality_codes_key"] != "")
    ].copy()

    return candidates

In [21]:
def make_stable_project_group_id(
    project_match_key: str,
    *,
    prefix: str = "project",
    length: int = 12,
) -> str:
    """
    Genera un identificador estable a partir de la clave de agrupación.
    """
    digest = hashlib.sha1(
        project_match_key.encode("utf-8")
    ).hexdigest()[:length]

    return f"{prefix}_{digest}"

In [22]:
def build_project_groups(
    project_candidates: pd.DataFrame,
) -> pd.DataFrame:
    """
    Construye la tabla gold de grupos de proyecto.

    Cada grupo representa un proyecto inferido de forma determinista a partir
    del nombre normalizado del activo y su conjunto de municipios INE.
    """
    required_cols = {
        "project_match_key",
        "asset_name",
        "asset_name_norm",
        "fecha_publicacion",
        "identificador_boe",
        "asset_mention_id",
    }
    validate_required_columns(project_candidates, required_cols)

    groups = (
        project_candidates
        .groupby("project_match_key", as_index=False)
        .agg(
            project_name=("asset_name", "first"),
            project_name_norm=("asset_name_norm", "first"),
            first_fecha_publicacion=("fecha_publicacion", "min"),
            last_fecha_publicacion=("fecha_publicacion", "max"),
            n_boe=("identificador_boe", "nunique"),
            n_asset_mentions=("asset_mention_id", "nunique"),
        )
    )

    groups["project_group_id"] = groups["project_match_key"].apply(
        make_stable_project_group_id
    )

    groups = groups[
        [
            "project_group_id",
            "project_match_key",
            "project_name",
            "project_name_norm",
            "first_fecha_publicacion",
            "last_fecha_publicacion",
            "n_boe",
            "n_asset_mentions",
        ]
    ]

    return groups.sort_values(
        ["project_name_norm", "first_fecha_publicacion"]
    ).reset_index(drop=True)

In [23]:
def build_project_asset_mentions(
    project_candidates: pd.DataFrame,
    project_groups: pd.DataFrame,
) -> pd.DataFrame:
    """
    Construye la tabla puente entre grupos de proyecto y menciones de activo.
    """
    required_candidate_cols = {
        "project_match_key",
        "asset_mention_id",
        "event_id",
        "identificador_boe",
        "fecha_publicacion",
        "asset_name",
        "asset_name_norm",
    }
    required_group_cols = {
        "project_group_id",
        "project_match_key",
    }

    validate_required_columns(project_candidates, required_candidate_cols)
    validate_required_columns(project_groups, required_group_cols)

    project_asset_mentions = project_candidates.merge(
        project_groups[["project_group_id", "project_match_key"]],
        on="project_match_key",
        how="left",
    )

    project_asset_mentions = project_asset_mentions[
        [
            "project_group_id",
            "project_match_key",
            "asset_mention_id",
            "event_id",
            "identificador_boe",
            "fecha_publicacion",
            "asset_name",
            "asset_name_norm",
        ]
    ].drop_duplicates()

    return project_asset_mentions.sort_values(
        [
            "project_group_id",
            "fecha_publicacion",
            "identificador_boe",
            "asset_mention_id",
        ]
    ).reset_index(drop=True)

In [24]:
asset_location_keys = build_asset_location_keys(asset_locations)

project_candidates = build_project_candidates(
    asset_mentions=asset_mentions,
    asset_location_keys=asset_location_keys,
)

project_groups = build_project_groups(project_candidates)

project_asset_mentions = build_project_asset_mentions(
    project_candidates=project_candidates,
    project_groups=project_groups,
)

In [ ]:
if project_asset_mentions["project_group_id"].isna().any():
    raise ValueError("Existen menciones de activo sin project_group_id.")

save_parquet(project_groups, PROJECT_GROUPS_PATH)
save_parquet(project_asset_mentions, PROJECT_ASSET_MENTIONS_PATH)

In [26]:
project_groups.loc[
    project_groups["project_name_norm"].str.contains(
        "badulaque",
        na=False,
    )
]

,project_group_id,project_match_key,project_name,project_name_norm,first_fecha_publicacion,last_fecha_publicacion,n_boe,n_asset_mentions
0,project_24aa6b49e6cc,parque eolico badulaque__15022|15025|15049|150...,Parque eólico Badulaque,parque eolico badulaque,2023-01-31,2023-04-28,2,2
